# 评估模块 — sub_pipeline

调用方式：在 main_pipeline.ipynb 中用 `%run` 执行本文件，之后可直接调用 `run_evaluation()`。

## 输入参数说明
- `df_actual`: 实际格局 OD DataFrame（列：o, d, 人数, distance）
- `df_ideal`: 极限格局 OD DataFrame
- `df_baseline`: 基准格局 OD DataFrame
- `df_random`: 随机格局 OD DataFrame
- `fence`: TAZ 边界 GeoDataFrame（含 taz, center_x, center_y 列）
- `output_section`: 输出章节路径（如 `3.Situation_Diagnosis/3.1Holistic_Diagnosis/3.1.1Static_Stats`）

In [ ]:
import sys
from pathlib import Path

project_root = Path(r"E:\00_Commute_Scenario_Research")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from src.config import get_result_path, STATIC_CSV, SHP_PATH, OD_CSV, VISUAL_CONFIG
from src.metrics_eval import (
    compute_taz_indicators,
    compute_balance_ratio,
    compute_street_self_sufficiency,
    compute_excess_commute,
    pattern_static_stats,
    pattern_flow_stats,
)
from src.visualization import (
    create_choropleth_map,
    create_diverging_map,
    create_street_choropleth,
    create_distance_pdf,
    create_distribution_plot,
    create_flowline,
)
from src.geo_excu import plot_std_ellipse
from src.utils import logger

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

STREET_SHP = Path(r"F:\02_250910_Commute\02-原始数据\[最终矫正版]街道乡镇边界\31\内部边界修改.shp")

## 静态结构层（3.1.1Static_Stats）

In [ ]:
def eval_M1_V1_people_distribution(df_actual, fence, output_section, save_fig=True):
    """M1: 人数分布统计 + V1: 人数分布图（O端、D端）"""
    gdf_o = compute_taz_indicators(df_actual, fence)

    if save_fig:
        out_o = get_result_path(output_section, 'star_人数分布_O端.png')
        create_choropleth_map(gdf_o, fence, '总通勤人数', 'total_people', out_o)

        # D端：按d聚合
        d_total = (df_actual.groupby('d')['人数'].sum()
                   .reset_index()
                   .rename(columns={'d': 'taz', '人数': '总通勤人数'}))
        fence_copy = fence[['taz', 'geometry']].copy()
        fence_copy['taz'] = fence_copy['taz'].astype(int)
        d_total['taz'] = d_total['taz'].astype(int)
        gdf_d = gpd.GeoDataFrame(
            fence_copy.merge(d_total, on='taz', how='left'),
            geometry='geometry', crs=fence.crs
        )
        out_d = get_result_path(output_section, 'star_人数分布_D端.png')
        create_choropleth_map(gdf_d, fence, '总通勤人数', 'total_people', out_d)

    return {'gdf_o': gdf_o}

In [ ]:
def eval_M3_V3_balance_ratio(fence, output_section, static_csv_path=None, save_fig=True):
    """M3: 职住平衡度 + V3: 平衡度分布图"""
    if static_csv_path is None:
        static_csv_path = STATIC_CSV

    gdf_balance = compute_balance_ratio(static_csv_path, fence)

    out_csv = get_result_path(output_section, 'star_平衡度指标.csv')
    gdf_balance[['taz', '居住人数', '工作人数', '平衡度']].to_csv(
        out_csv, index=False, encoding='utf-8-sig')

    if save_fig:
        out_fig = get_result_path(output_section, 'star_平衡度分布图.png')
        create_choropleth_map(gdf_balance, fence, '平衡度', 'balance_ratio', out_fig)

    return gdf_balance

In [ ]:
def eval_M4_V4_street_self_sufficiency(df_actual, fence, output_section,
                                        street_shp_path=None, save_fig=True):
    """M4: 街道级自给度 + V4: 自给度分布图"""
    if street_shp_path is None:
        street_shp_path = STREET_SHP

    gdf_street = compute_street_self_sufficiency(df_actual, fence, street_shp_path)

    out_csv = get_result_path(output_section, 'star_自给度指标_街道.csv')
    gdf_street[['street_idx', '内部通勤人数', '总通勤人数', '自给度']].to_csv(
        out_csv, index=False, encoding='utf-8-sig')

    if save_fig:
        gdf_taz_boundary = fence.copy()
        if gdf_taz_boundary.crs != gdf_street.crs:
            gdf_taz_boundary = gdf_taz_boundary.to_crs(gdf_street.crs)
        out_fig = get_result_path(output_section, 'star_自给度分布图_街道.png')
        create_street_choropleth(gdf_street, gdf_taz_boundary, '自给度', 'self_sufficiency', out_fig)

    return gdf_street

In [ ]:
def eval_V92_std_ellipse(fence, output_section, static_csv_path=None, save_fig=True):
    """V92: 标准差椭圆（O端居住地 + D端工作地）"""
    if static_csv_path is None:
        static_csv_path = STATIC_CSV

    if save_fig:
        out_fig = get_result_path(output_section, 'star_标准差椭圆.png')
        plot_std_ellipse(fence, static_csv_path, out_fig)

    return {}

## 优化潜力层（3.1.3Pattern_Comparison）

In [ ]:
def eval_M5_M8_excess_commute(df_actual, output_section,
                               c_min_km=1.118, c_ran_km=13.144):
    """M5-M8: 超额通勤指标（EC, NEC, CE, NCE）"""
    df_valid = df_actual[df_actual['distance'].notna() & (df_actual['人数'] > 0)].copy()
    c_obs_m = (df_valid['人数'] * df_valid['distance']).sum() / df_valid['人数'].sum()
    c_obs_km = c_obs_m / 1000.0

    out_dir = get_result_path(output_section, '').parent
    result = compute_excess_commute(c_obs_km, c_min_km, c_ran_km, output_dir=out_dir)

    logger.info(
        f"超额通勤: C_obs={c_obs_km:.3f}km, "
        f"EC={result['EC(%)']:.1f}%, NEC={result['NEC(%)']:.1f}%"
    )
    return result

In [ ]:
def eval_V96_static_diff_maps(df_actual, df_compare, name_compare, fence,
                               output_section, save_fig=True):
    """V96: 静态分布人数差值图（O端、D端）"""
    def _get_static(df, end='o'):
        col = 'o' if end == 'o' else 'd'
        return (df.groupby(col)['人数'].sum()
                .reset_index()
                .rename(columns={col: 'taz', '人数': '人数'}))

    fence_copy = fence[['taz', 'geometry']].copy()
    fence_copy['taz'] = fence_copy['taz'].astype(int)

    for end in ['O端', 'D端']:
        key = 'o' if end == 'O端' else 'd'
        a_df = _get_static(df_actual, key).rename(columns={'人数': '人数_actual'})
        b_df = _get_static(df_compare, key).rename(columns={'人数': f'人数_{name_compare}'})

        merged = a_df.merge(b_df, on='taz', how='outer').fillna(0)
        merged['taz'] = merged['taz'].astype(int)
        merged['人数差值'] = merged['人数_actual'] - merged[f'人数_{name_compare}']

        gdf_diff = gpd.GeoDataFrame(
            fence_copy.merge(merged, on='taz', how='left'),
            geometry='geometry', crs=fence.crs
        )

        if save_fig:
            out_fig = get_result_path(
                output_section, f'star_静态分布差_实际vs{name_compare}_{end}.png')
            create_diverging_map(gdf_diff, fence, '人数差值', 'diff_people', out_fig)

    return {}

In [ ]:
def eval_V97_distance_pdf(df_actual, df_ideal, df_baseline, df_random,
                           output_section, save_fig=True):
    """V97: 通勤距离PDF对比图（四格局合并到一张图）"""
    if save_fig:
        out_fig = get_result_path(output_section, 'star_格局对比_距离pdf_四格局.png')
        create_distance_pdf(
            df_list=[df_actual, df_ideal, df_baseline, df_random],
            names=['实际格局', '极限格局', '基准格局', '随机格局'],
            output_path=out_fig,
            col='distance',
            weight_col='人数',
            cap_abs=30000,
            unit_scale=1/1000,
        )
    return {}

In [ ]:
def eval_V98_flow_boxplot(df_actual, df_ideal, df_baseline, df_random,
                           output_section, save_fig=True):
    """V98: 人流箱线图（四格局合并到一张图，双子图：左细节/右对数轴）"""
    if save_fig:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(24, 7))

        cap_left = 200.0
        cap_right = 2000.0
        data_list_left = []
        data_list_right = []
        labels = []
        for df, name in [
            (df_actual,   '实际格局'),
            (df_ideal,    '极限格局'),
            (df_baseline, '基准格局'),
            (df_random,   '随机格局'),
        ]:
            s = df['人数'].dropna()
            data_list_left.append(s[s <= cap_left].values)
            data_list_right.append(s[s <= cap_right].values)
            labels.append(name)

        colors_bp = ['#2E7D9A', '#C65A4A', '#4A8F4A', '#8B6914']

        # 左图：0-200细节
        bp1 = ax1.boxplot(
            data_list_left,
            tick_labels=labels,
            patch_artist=True,
            vert=False,
            medianprops=dict(color='black', linewidth=2),
            whiskerprops=dict(linewidth=1.5),
            capprops=dict(linewidth=1.5),
            flierprops=dict(marker='o', markersize=3, alpha=0.4),
        )
        for patch, color in zip(bp1['boxes'], colors_bp):
            patch.set_facecolor(color)
            patch.set_alpha(0.75)
        ax1.set_xlabel('OD对通勤人数（人）', fontsize=22, fontfamily='SimHei')
        ax1.set_title(f'细节视图（0-{int(cap_left)}人）', fontsize=20, fontfamily='SimHei')
        ax1.tick_params(axis='both', labelsize=20)
        for label in ax1.get_yticklabels():
            label.set_fontfamily('SimHei')
        ax1.grid(True, axis='x', alpha=0.3)

        # 右图：对数轴全貌
        bp2 = ax2.boxplot(
            data_list_right,
            tick_labels=labels,
            patch_artist=True,
            vert=False,
            medianprops=dict(color='black', linewidth=2),
            whiskerprops=dict(linewidth=1.5),
            capprops=dict(linewidth=1.5),
            flierprops=dict(marker='o', markersize=3, alpha=0.4),
        )
        for patch, color in zip(bp2['boxes'], colors_bp):
            patch.set_facecolor(color)
            patch.set_alpha(0.75)
        ax2.set_xscale('log')
        ax2.set_xlabel('OD对通勤人数（人，对数轴）', fontsize=22, fontfamily='SimHei')
        ax2.set_title(f'全貌视图（对数轴，上限{int(cap_right)}人）', fontsize=20, fontfamily='SimHei')
        ax2.tick_params(axis='both', labelsize=20)
        for label in ax2.get_yticklabels():
            label.set_fontfamily('SimHei')
        ax2.grid(True, axis='x', alpha=0.3)

        plt.tight_layout()

        out_fig = get_result_path(output_section, 'star_格局对比_人流箱线图_四格局_v3.png')
        plt.savefig(out_fig, dpi=VISUAL_CONFIG['dpi'], bbox_inches='tight', facecolor='white')
        plt.close(fig)
        logger.info(f"人流箱线图（双子图）已保存: {out_fig}")

    return {}

In [ ]:
def eval_V94_od_flowline_diff(df_actual, df_compare, name_compare, fence,
                               output_section, save_fig=True):
    """V94: OD流量差值流线图"""
    if save_fig:
        df_merged = (
            df_actual[['o', 'd', '人数']]
            .merge(
                df_compare[['o', 'd', '人数']].rename(columns={'人数': '人数_compare'}),
                on=['o', 'd'], how='outer'
            )
            .fillna(0)
        )
        df_merged['差值'] = df_merged['人数'] - df_merged['人数_compare']

        out_fig = get_result_path(output_section, f'star_OD流量差_实际vs{name_compare}.png')
        create_flowline(df_merged, fence, out_fig, flow_col='差值', is_diff=True)

    return {}

## 注册表与主函数

In [ ]:
from src.metrics_eval import (
    compute_time_indicators,
    compute_transport_mode_stats,
    compute_street_balance_ratio,
)
from src.visualization import (
    create_pie_chart,
    create_blank_taz_map,
    create_od_flowmap_tbd,
)
from src.geo_excu import plot_std_ellipse_flow, community_detection_tbd


def eval_M9_V9_time_distribution(df_od_full, fence, output_section, save_fig=True):
    """M9: 通勤时间指标 + V9: 时间分布图（TAZ地图 + 饼图）"""
    gdf_time, time_stats = compute_time_indicators(df_od_full, fence)

    out_csv = get_result_path(output_section, 'star_动态行为指标_时间.csv')
    gdf_time[['taz', '平均通勤时间_min']].to_csv(out_csv, index=False, encoding='utf-8-sig')

    if save_fig:
        # TAZ级时间地图
        out_map = get_result_path(output_section, 'star_平均通勤时间分布图_TAZ.png')
        from src.visualization import create_choropleth_map
        create_choropleth_map(gdf_time, fence, '平均通勤时间_min', 'avg_time', out_map)

        # 时长区间饼图
        pie_data = {k: v for k, v in time_stats.items() if k != '全局平均时间_min'}
        out_pie = get_result_path(output_section, 'star_通勤时长区间占比_饼图.png')
        create_pie_chart(pie_data, out_pie, title='通勤时长区间占比')

    return {'gdf_time': gdf_time, 'time_stats': time_stats}


def eval_M10_V10_transport_mode(df_od_full, output_section, save_fig=True):
    """M10: 交通方式统计 + V10: 交通方式饼图"""
    mode_stats = compute_transport_mode_stats(df_od_full)

    out_csv = get_result_path(output_section, 'star_动态行为指标_交通方式.csv')
    import pandas as pd
    pd.DataFrame([mode_stats]).to_csv(out_csv, index=False, encoding='utf-8-sig')

    if save_fig:
        out_pie = get_result_path(output_section, 'star_交通方式占比_饼图.png')
        create_pie_chart(mode_stats, out_pie, title='交通方式占比')

    return mode_stats


def eval_V11_od_flowmap(df_od_full, fence, output_section, mapbox_token, bounds,
                         zoom=14, style=6, top_n=500, save_fig=True):
    """V11: OD流线图（transbigdata交互式HTML + PNG）"""
    if save_fig:
        out_html = get_result_path(output_section, 'star_OD流线图_实际格局.html')
        create_od_flowmap_tbd(
            df_od=df_od_full,
            fence=fence,
            output_path=out_html,
            mapbox_token=mapbox_token,
            bounds=bounds,
            zoom=zoom,
            style=style,
            top_n=top_n,
        )
    return {}


def eval_V12_community_detection(df_od_full, fence, output_section, mapbox_token, bounds,
                                   zoom=14, style=6, save_fig=True):
    """V12: 社区发现（igraph Louvain）"""
    if save_fig:
        out_html = get_result_path(output_section, 'star_社区发现结果.html')
        result = community_detection_tbd(
            df_od=df_od_full,
            fence=fence,
            output_path=out_html,
            mapbox_token=mapbox_token,
            bounds=bounds,
            zoom=zoom,
            style=style,
        )
        return result
    return {}


def eval_V13_std_ellipse_flow(df_od_full, fence, output_section, save_fig=True):
    """V13: 基于OD流的标准差椭圆"""
    if save_fig:
        out_fig = get_result_path(output_section, 'star_标准差椭圆_通勤流.png')
        result = plot_std_ellipse_flow(df_od_full, fence, out_fig)
        return result
    return {}

In [ ]:
EVAL_REGISTRY = {
    "M1_V1_people_distribution":     eval_M1_V1_people_distribution,
    "M3_V3_balance_ratio":           eval_M3_V3_balance_ratio,
    "M4_V4_street_self_sufficiency": eval_M4_V4_street_self_sufficiency,
    "V92_std_ellipse":               eval_V92_std_ellipse,
    "M5_M8_excess_commute":          eval_M5_M8_excess_commute,
    "V96_static_diff_maps":          eval_V96_static_diff_maps,
    "V97_distance_pdf":              eval_V97_distance_pdf,
    "V98_flow_boxplot":              eval_V98_flow_boxplot,
    "V94_od_flowline_diff":          eval_V94_od_flowline_diff,
    "M9_V9_time_distribution":       eval_M9_V9_time_distribution,
    "M10_V10_transport_mode":        eval_M10_V10_transport_mode,
    "V11_od_flowmap":                eval_V11_od_flowmap,
    "V12_community_detection":       eval_V12_community_detection,
    "V13_std_ellipse_flow":          eval_V13_std_ellipse_flow,
}

EVAL_CONFIGS = {
    "static_structure": [
        "M1_V1_people_distribution",
        "M3_V3_balance_ratio",
        "M4_V4_street_self_sufficiency",
        "V92_std_ellipse",
    ],
    "pattern_comparison": [
        "M5_M8_excess_commute",
        "V97_distance_pdf",
        "V98_flow_boxplot",
    ],
    "dynamic_behavior": [
        "M9_V9_time_distribution",
        "M10_V10_transport_mode",
        "V11_od_flowmap",
        "V12_community_detection",
        "V13_std_ellipse_flow",
    ],
    "holistic_diagnosis": [
        "M1_V1_people_distribution",
        "M3_V3_balance_ratio",
        "M4_V4_street_self_sufficiency",
        "V92_std_ellipse",
        "M5_M8_excess_commute",
        "V97_distance_pdf",
        "V98_flow_boxplot",
    ],
}


def run_evaluation(
    fence,
    output_section,
    df_actual=None,
    df_ideal=None,
    df_baseline=None,
    df_random=None,
    df_od_full=None,
    config_name=None,
    items=None,
    save_fig=True,
    static_csv_path=None,
    street_shp_path=None,
    c_min_km=1.118,
    c_ran_km=13.144,
    mapbox_token=None,
    bounds=None,
    zoom=14,
    style=6,
):
    """
    评估主函数，支持两种调用方式：
    - config_name: 使用预设配置（'static_structure', 'pattern_comparison', 'dynamic_behavior', 'holistic_diagnosis'）
    - items: 直接指定要运行的编号列表

    Args:
        fence:            TAZ 边界 GeoDataFrame（含 taz, center_x, center_y 列）
        output_section:   输出章节路径（传入 get_result_path 的 section 参数）
        df_actual:        实际格局 OD DataFrame（列：o, d, 人数, distance）
        df_ideal:         极限格局 OD DataFrame
        df_baseline:      基准格局 OD DataFrame
        df_random:        随机格局 OD DataFrame
        df_od_full:       完整OD数据（含时间、交通方式列）
        config_name:      预设配置名
        items:            指定运行的评估项列表
        save_fig:         是否保存图片（False 时只算指标，跳过出图）
        static_csv_path:  static.csv 路径，None 时使用 config.STATIC_CSV
        street_shp_path:  街道边界 shp 路径，None 时使用模块默认值
        c_min_km:         极限格局平均通勤距离（km），默认 1.118
        c_ran_km:         随机格局平均通勤距离（km），默认 13.144
        mapbox_token:     Mapbox访问令牌（动态行为层需要）
        bounds:           地图范围 [lon_min, lat_min, lon_max, lat_max]
        zoom:             地图缩放级别
        style:            transbigdata地图样式编号

    Returns:
        dict: {item_id: result}
    """
    if items is None:
        if config_name is None:
            raise ValueError("config_name 和 items 至少提供一个")
        items = EVAL_CONFIGS[config_name]

    results = {}
    for item_id in items:
        if item_id not in EVAL_REGISTRY:
            logger.warning(f"[run_evaluation] 未知评估项: {item_id}，跳过")
            continue

        logger.info(f"[run_evaluation] 执行: {item_id}")

        if item_id == "M1_V1_people_distribution":
            results[item_id] = eval_M1_V1_people_distribution(
                df_actual, fence, output_section, save_fig)

        elif item_id == "M3_V3_balance_ratio":
            results[item_id] = eval_M3_V3_balance_ratio(
                fence, output_section, static_csv_path, save_fig)

        elif item_id == "M4_V4_street_self_sufficiency":
            results[item_id] = eval_M4_V4_street_self_sufficiency(
                df_actual, fence, output_section, street_shp_path, save_fig)

        elif item_id == "V92_std_ellipse":
            results[item_id] = eval_V92_std_ellipse(
                fence, output_section, static_csv_path, save_fig)

        elif item_id == "M5_M8_excess_commute":
            results[item_id] = eval_M5_M8_excess_commute(
                df_actual, output_section, c_min_km, c_ran_km)

        elif item_id == "V96_static_diff_maps":
            for df_cmp, name_cmp in [
                (df_ideal,    '极限格局'),
                (df_baseline, '基准格局'),
                (df_random,   '随机格局'),
            ]:
                if df_cmp is not None:
                    eval_V96_static_diff_maps(
                        df_actual, df_cmp, name_cmp, fence, output_section, save_fig)
            results[item_id] = {}

        elif item_id == "V97_distance_pdf":
            results[item_id] = eval_V97_distance_pdf(
                df_actual, df_ideal, df_baseline, df_random, output_section, save_fig)

        elif item_id == "V98_flow_boxplot":
            results[item_id] = eval_V98_flow_boxplot(
                df_actual, df_ideal, df_baseline, df_random, output_section, save_fig)

        elif item_id == "V94_od_flowline_diff":
            for df_cmp, name_cmp in [
                (df_ideal,    '极限格局'),
                (df_baseline, '基准格局'),
                (df_random,   '随机格局'),
            ]:
                if df_cmp is not None:
                    eval_V94_od_flowline_diff(
                        df_actual, df_cmp, name_cmp, fence, output_section, save_fig)
            results[item_id] = {}

        elif item_id == "M9_V9_time_distribution":
            results[item_id] = eval_M9_V9_time_distribution(
                df_od_full, fence, output_section, save_fig)

        elif item_id == "M10_V10_transport_mode":
            results[item_id] = eval_M10_V10_transport_mode(
                df_od_full, output_section, save_fig)

        elif item_id == "V11_od_flowmap":
            results[item_id] = eval_V11_od_flowmap(
                df_od_full, fence, output_section, mapbox_token, bounds, zoom, style, save_fig=save_fig)

        elif item_id == "V12_community_detection":
            results[item_id] = eval_V12_community_detection(
                df_od_full, fence, output_section, mapbox_token, bounds, zoom, style, save_fig)

        elif item_id == "V13_std_ellipse_flow":
            results[item_id] = eval_V13_std_ellipse_flow(
                df_od_full, fence, output_section, save_fig)

    logger.info(f"[run_evaluation] 完成，共执行 {len(results)} 项")
    return results

## 使用示例（在 main_pipeline.ipynb 中调用）

In [ ]:
# ============================================================
# 在 main_pipeline.ipynb 中的调用示例（此 cell 仅供参考，不执行）
# ============================================================

# Step 1: 加载评估模块
# %run notebooks/sub_pipeline/evaluation.ipynb

# Step 2A: 静态结构层
# results_static = run_evaluation(
#     fence=fence,
#     output_section='3.Situation_Diagnosis/3.1Holistic_Diagnosis/3.1.1Static_Stats',
#     df_actual=df_actual,
#     config_name='static_structure',
# )

# Step 2B: 优化潜力层（需要四个格局）
# results_pattern = run_evaluation(
#     fence=fence,
#     output_section='3.Situation_Diagnosis/3.1Holistic_Diagnosis/3.1.3Pattern_Comparison',
#     df_actual=df_actual,
#     df_ideal=df_ideal,
#     df_baseline=df_baseline_int,
#     df_random=df_random_int,
#     config_name='pattern_comparison',
# )

# Step 2C: 差值图（单独调用）
# run_evaluation(
#     fence=fence,
#     output_section='3.Situation_Diagnosis/3.1Holistic_Diagnosis/3.1.3Pattern_Comparison',
#     df_actual=df_actual,
#     df_ideal=df_ideal,
#     df_baseline=df_baseline_int,
#     df_random=df_random_int,
#     items=['V96_static_diff_maps', 'V94_od_flowline_diff'],
# )

# Step 2D: 只算指标不出图（快速验证）
# run_evaluation(
#     fence=fence,
#     output_section='...',
#     df_actual=df_actual,
#     config_name='static_structure',
#     save_fig=False,
# )